# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [8]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

bins = [-1, 30, 90, 100000]
labels = ["0-30d", "31-90d", "90+d"]
df["staleness_bucket"] = pd.cut(df["days_since_last_update"], bins=bins, labels=labels)
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

t1 = df.groupby("staleness_bucket", observed=True).agg(
    n=("content_id", "count"),
    pct_declining=("is_declining_label", "mean")
)
print(t1)

d2 = df[df["avg_position"] > 0].copy()
d2["position_bucket"] = pd.cut(d2["avg_position"], bins=[0,10,20,1000], labels=["top10","11-20","20+"])

t2 = d2.groupby("position_bucket", observed=True).agg(
    n=("content_id", "count"),
    mean_ctr=("ctr", "mean")
)
print(t2)

                      n  pct_declining
staleness_bucket                      
0-30d             20480       0.511377
31-90d              175       0.588571
90+d               9345       0.608454
                     n  mean_ctr
position_bucket                 
top10            12983  0.832373
11-20             7273  0.323443
20+               8539  0.211333


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [18]:
df["visible"] = (df["impressions_90d"] >= 500).astype(int)
df["slip"] = ((df["avg_position"] > 20) & (df["avg_position"] > 0)).astype(int)
df["stale"] = (df["days_since_last_update"] >= 90).astype(int)

# position slip weighted higher than staleness — staleness alone was a weak signal (Cell 1)
df["score"] = df["visible"] * (df["slip"]*df["impressions_90d"] + df["stale"]*df["impressions_90d"]*0.3)

def reason(r):
    if r["slip"] and r["stale"]: return "STALE_AND_SLIPPING"
    if r["slip"]: return "POSITION_SLIP_ONLY"
    if r["stale"] and r["visible"]: return "STALE_HIGH_VISIBILITY"
    return "LOW_PRIORITY"

df["reason_code"] = df.apply(reason, axis=1)
df["action"] = df["score"].apply(lambda s: "REFRESH" if s > 0 else "NO_ACTION")

queue = df.sort_values("score", ascending=False).reset_index(drop=True)
queue[["content_id","score","reason_code","action"]].to_csv("../../work/outputs/baseline_action_score.csv", index=False)
queue[["content_id","score","reason_code","action"]].head(20)

,content_id,score,reason_code,action
0,content_2dba2b1f9536,576464.2,STALE_AND_SLIPPING,REFRESH
1,content_2cb567c3c89b,497727.0,POSITION_SLIP_ONLY,REFRESH
2,content_b28d1efd668f,372590.4,STALE_AND_SLIPPING,REFRESH
3,content_813e88069237,303629.3,STALE_AND_SLIPPING,REFRESH
4,content_b511d4bc4ad2,267689.5,STALE_AND_SLIPPING,REFRESH
5,content_f02b48f88241,235968.2,STALE_AND_SLIPPING,REFRESH
6,content_05e9b4cd9ccf,232702.6,STALE_AND_SLIPPING,REFRESH
7,content_ff94c9b6b411,228566.0,POSITION_SLIP_ONLY,REFRESH
8,content_66b4046cc144,217415.0,POSITION_SLIP_ONLY,REFRESH
9,content_a023517539fe,214047.0,POSITION_SLIP_ONLY,REFRESH


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

1. content_2dba2b1f9536 — REFRESH — STALE_AND_SLIPPING, highest score from very high impressions with both stale content and slipped position — wrong if this page was intentionally deprioritized or is seasonal
2. content_2cb567c3c89b — REFRESH — POSITION_SLIP_ONLY, position slip is the sole driver, staleness not flagged — wrong if the position drop is a temporary SERP fluctuation, not real decline
3. content_b28d1efd668f — REFRESH — STALE_AND_SLIPPING, both signals present at high impressions — wrong if the page's traffic is naturally seasonal/cyclical
4. content_813e88069237 — REFRESH — STALE_AND_SLIPPING — wrong if recently updated but the update isn't yet reflected in days_since_last_update
5. content_b511d4bc4ad2 — REFRESH — STALE_AND_SLIPPING — wrong if this is evergreen content where position naturally fluctuates without real decline
6. content_f02b48f88241 — REFRESH — STALE_AND_SLIPPING — wrong if impressions are mostly branded/navigational queries unaffected by staleness
7. content_05e9b4cd9ccf — REFRESH — STALE_AND_SLIPPING — wrong if content_type has known missing/weak keyword data inflating apparent risk
8. content_ff94c9b6b411 — REFRESH — POSITION_SLIP_ONLY, no staleness support — wrong if position slip is due to a SERP feature (e.g. featured snippet) stealing clicks, not real ranking loss
9. content_66b4046cc144 — REFRESH — POSITION_SLIP_ONLY — wrong if this keyword has high competition and the slip reflects market shift, not something a refresh fixes
10. content_a023517539fe — REFRESH — POSITION_SLIP_ONLY — wrong if avg_position is close to the 20-bucket boundary (e.g. ≈21) and the slip is marginal, not severe

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak pick: content_ff94c9b6b411 (row 8) — POSITION_SLIP_ONLY with no staleness signal at all. Given Cell 1 showed staleness itself is only a weak/MIXED signal, relying purely on position slip without staleness support is the shakiest basis in the top 10 — the rule can't distinguish a genuine ranking loss from short-term SERP noise here.

Leakage check: score formula (Cell 2) uses only impressions_90d, avg_position, days_since_last_update — no use of trend_pct, trend_direction, or is_declining_label. is_declining_label was used only in Cell 1 for the staleness verdict diagnostic, never as a scoring input.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.